# ASR Baseline (Whisper) – Zeroth Korean

Zeroth Korean **test set**을 기반으로 Whisper ASR 베이스라인을 빠르게 확인하기 위한 노트북입니다.

- 입력 오디오: `/home/data/data/zeroth/test_data_01/**/**/*.flac` (하위 디렉터리를 포함한 모든 flac 파일)
- 정답 스크립트: `/home/data/data/zeroth/test_data_01/refs.csv` (파일명 기준으로 오디오와 매칭)

---

## 목적
1) **Whisper 모델별 성능(WER/CER) 및 실행시간 저장**(경진대회/수업 실험 기록)

---

## 라이선스/고지 (Zeroth Korean)
- 본 노트북은 **Zeroth Korean 데이터셋의 test set**을 실험용으로 사용합니다.
- 데이터셋의 원본 라이선스 및 이용조건은 데이터 제공처/원본 문서를 따릅니다.
- 본 실험 결과(전사/점수)는 교육 목적의 벤치마크 기록이며, 데이터 재배포는 하지 않습니다.

In [ ]:
import os, time, json
from pathlib import Path

import pandas as pd
import torch
from accelerate import Accelerator
from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
from jiwer import wer, cer


In [ ]:
from pathlib import Path
import os

HOME = "/home/data"
BASE_DIR = Path("/home/data")
DATA_AUDIO_DIR = "/home/data/data/zeroth/test_data_01"
REF_CSV = "/home/data/data/zeroth/test_data_01/refs.csv"

# 여러 모델을 여기서 바꿔가며 비교
MODEL_IDS = [
    "openai/whisper-tiny",
    "openai/whisper-base",
    "openai/whisper-small",
    # "openai/whisper-medium",  # 느리면 주석
]

LANGUAGE = "korean"
TASK = "transcribe"

MAX_FILES = int(os.environ.get("MAX_FILES", "50"))
SEED = int(os.environ.get("SEED", "42"))

OUT_DIR = Path(os.environ.get("OUT_DIR", BASE_DIR / "expr" / "week2" / "02-outputs_asr"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_AUDIO_DIR:", DATA_AUDIO_DIR)
print("REF_CSV:", REF_CSV)
print("MODELS:", MODEL_IDS)
print("MAX_FILES:", MAX_FILES, "| SEED:", SEED)

assert Path(DATA_AUDIO_DIR).exists(), f"Missing audio dir: {DATA_AUDIO_DIR}"
assert Path(REF_CSV).exists(), f"Missing refs.csv: {REF_CSV}"

In [ ]:
df_ref = pd.read_csv(REF_CSV, sep='\t')

# refs.csv 컬럼: file/ref로 통일
if "ref" not in df_ref.columns and "transcript" in df_ref.columns:
    df_ref = df_ref.rename(columns={"transcript": "ref"})

assert "file" in df_ref.columns, f"refs.csv needs 'file' column. got={df_ref.columns.tolist()}"
assert "ref" in df_ref.columns, f"refs.csv needs 'ref' (or transcript) column. got={df_ref.columns.tolist()}"

# file은 basename으로 맞추기
df_ref["file"] = df_ref["file"].astype(str).apply(lambda x: os.path.basename(x))
df_ref["ref"] = df_ref["ref"].astype(str)

df_ref.head()


In [ ]:
import random, glob

audio_files_all = sorted(Path(p) for p in glob.glob(os.path.join(DATA_AUDIO_DIR, "**", "*.flac"), recursive=True))
assert len(audio_files_all) > 0, f"No wav files in {DATA_AUDIO_DIR}"

# 2) basename -> fullpath 매핑 (중복 basename이 있으면 마지막이 덮어씀; Zeroth는 보통 유니크)
wav_map = {p.name: str(p) for p in audio_files_all}

# refs.csv에 있는 파일만 대상으로 (정답 없는 파일 제외)
ref_set = set(df_ref["file"].tolist())
audio_files = [p for p in audio_files_all if p.name in ref_set]

assert len(audio_files) > 0, "No overlap between refs.csv and audio flac files."

# 빠른 확인용 샘플링 (재현 가능). 직접 실험해 볼때는 사용하지 않는 코드입니다.
#rng = random.Random(SEED)
#if MAX_FILES > 0 and len(audio_files) > MAX_FILES:
#    audio_files = rng.sample(audio_files, MAX_FILES)

audio_files = sorted(audio_files)  # 출력 보기 좋게
print("#audio used:", len(audio_files), "/", len(audio_files_all))
print("sample:", audio_files[0])


In [ ]:
df_eval_by_model = {}  # model_id -> df_eval (columns: file, ref, hyp, model_id, wav_path)
df_hyp_by_model  = {}  # model_id -> df_hyp (columns: file, hyp, model_id)


In [ ]:
def transcribe_one_model(model_id: str, files, df_ref: pd.DataFrame):
    import os, time
    from pathlib import Path
    import pandas as pd
    import torch
    from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
    from jiwer import wer, cer

    BATCH_SIZE = 8

    # device/dtype (함수 내부에서 확정)
    if torch.cuda.is_available():
        device = 0
        visible = os.environ.get("CUDA_VISIBLE_DEVICES", "(not set)")
        dtype = torch.float16
    else:
        device = -1
        visible = "(cpu)"
        dtype = torch.float32

    print(f"MODEL: {model_id} | CUDA_VISIBLE_DEVICES: {visible} | pipeline device: {device} | dtype: {dtype}")

    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, dtype=dtype)
    if torch.cuda.is_available():
        model = model.to("cuda")

    asr = hf_pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        device=device,
    )

    # files: Path 리스트든 str 리스트든 모두 처리
    file_strs = [str(p) for p in files]

    t0 = time.time()
    outs = asr(
        file_strs,
        batch_size=BATCH_SIZE,
        generate_kwargs={"language": LANGUAGE, "task": TASK, "temperature": 0.0},
    )
    elapsed = round(time.time() - t0, 2)

    df_hyp = pd.DataFrame(
        [{"file": os.path.basename(p), "hyp": o["text"].strip()} for p, o in zip(file_strs, outs)]
    )
    df_hyp["model_id"] = model_id

    # merge + 매칭 체크
    df_eval = df_ref.merge(df_hyp, on="file", how="inner")
    df_eval["model_id"] = model_id

    print("ref rows:", len(df_ref), "| hyp rows:", len(df_hyp), "| matched:", len(df_eval))
    if len(df_eval) == 0:
        raise ValueError("No matched rows. Check df_ref['file'] == basename(wav filename).")

    w = wer(df_eval["ref"].tolist(), df_eval["hyp"].tolist())
    c = cer(df_eval["ref"].tolist(), df_eval["hyp"].tolist())

    return df_hyp, df_eval, elapsed, w, c


In [ ]:
all_results = []

for model_id in MODEL_IDS:
    print("\n" + "="*80)
    print("MODEL:", model_id)

    df_hyp, df_eval, elapsed, w, c = transcribe_one_model(model_id, audio_files, df_ref)

    # 저장(메모리)
    df_hyp_by_model[model_id]  = df_hyp
    df_eval_by_model[model_id] = df_eval

    # 저장(파일) - 선택
    hyp_path = OUT_DIR / f"hyp_{model_id.replace('/','_')}.csv"
    eval_path = OUT_DIR / f"eval_{model_id.replace('/','_')}.csv"
    df_hyp.to_csv(hyp_path, index=False)
    df_eval.to_csv(eval_path, index=False)

    row = {
        "model_id": model_id,
        "n_files": len(audio_files),
        "elapsed_sec": round(elapsed, 3),
        "sec_per_file": round(elapsed / max(len(audio_files), 1), 4),
        "wer": round(w, 6),
        "cer": round(c, 6),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    all_results.append(row)

    print("Saved hyp:", hyp_path)
    print("Saved eval:", eval_path)
    print("WER:", row["wer"], "| CER:", row["cer"], "| elapsed(s):", row["elapsed_sec"])


In [ ]:
import pandas as pd
import os
from pathlib import Path

# 1) 모델별 df_eval들을 하나로 합치기 (long)
df_all = pd.concat(df_eval_by_model.values(), ignore_index=True)

# 안전장치: file은 basename 통일
df_all["file"] = df_all["file"].astype(str).apply(os.path.basename)
df_all["ref"]  = df_all["ref"].astype(str)
df_all["hyp"]  = df_all["hyp"].astype(str)

# 2) wide(파일 단위 비교)로 변환: (file, ref) 한 줄 + 모델별 hyp 컬럼
df_cmp = (
    df_all.pivot_table(
        index=["file", "ref"],
        columns="model_id",
        values="hyp",
        aggfunc="first"
    )
    .reset_index()
)

# 3) 컬럼명 정리: hyp__openai_whisper_base 형태
df_cmp.columns = [
    f"hyp__{c.replace('/', '_')}" if c not in ("file", "ref") else c
    for c in df_cmp.columns
]

# 3) df_cmp_view (또는 df)에 wav_path를 "실제 경로"로 채움
df_cmp_view = df_cmp.copy()
df_cmp_view["wav_path"] = df_cmp_view["file"].map(wav_map)

# 4) 검증
missing = df_cmp_view["wav_path"].isna().sum()
print("missing wav_path:", missing, "/", len(df_cmp_view))
print("example wav_path:", df_cmp_view["wav_path"].dropna().iloc[0])


# 4) 오디오 경로 붙이기
DATA_AUDIO_DIR_P = Path(DATA_AUDIO_DIR)  # DATA_AUDIO_DIR가 Path면 그대로 써도 됨
df_cmp["wav_path"] = df_cmp["file"].astype(str).apply(lambda fn: str(DATA_AUDIO_DIR_P / fn))

print("rows:", len(df_cmp))
print("cols:", df_cmp.columns.tolist())
df_cmp.head(10)

In [ ]:
import pandas as pd
from jiwer import wer, cer

# df_cmp: [file, ref, wav_path, hyp__...] 형태여야 함
hyp_cols = [c for c in df_cmp.columns if c.startswith("hyp__")]
assert len(hyp_cols) > 0, "hyp__ 컬럼이 없습니다. df_cmp pivot 셀부터 확인하세요."

def safe_wer(ref, hyp):
    ref = "" if pd.isna(ref) else str(ref)
    hyp = "" if pd.isna(hyp) else str(hyp)
    return wer([ref], [hyp])

def safe_cer(ref, hyp):
    ref = "" if pd.isna(ref) else str(ref)
    hyp = "" if pd.isna(hyp) else str(hyp)
    return cer([ref], [hyp])

# (주의) apply row-wise는 느릴 수 있음. 지금 50개면 충분히 OK.
for hyp_c in hyp_cols:
    tag = hyp_c.replace("hyp__", "")          # openai_whisper_base 같은 형태
    df_cmp[f"wer__{tag}"] = df_cmp.apply(lambda r: safe_wer(r["ref"], r[hyp_c]), axis=1)
    df_cmp[f"cer__{tag}"] = df_cmp.apply(lambda r: safe_cer(r["ref"], r[hyp_c]), axis=1)

# 평균 CER 기준으로 모델 정렬
cer_cols = [c for c in df_cmp.columns if c.startswith("cer__")]
model_order = sorted(cer_cols, key=lambda cc: df_cmp[cc].mean())

# 보기용 컬럼 재배열: file/ref/wav_path + (모델별 cer/wer/hyp 묶음)
base_cols = ["file", "ref", "wav_path"]
cols = base_cols.copy()

for cer_c in model_order:
    tag = cer_c.replace("cer__", "")
    wer_c = f"wer__{tag}"
    hyp_c = f"hyp__{tag}"
    cols += [cer_c, wer_c, hyp_c]

df_cmp_view = df_cmp[cols].copy()

# 요약(모델별 평균 WER/CER)
summary = []
for cer_c in model_order:
    tag = cer_c.replace("cer__", "")
    summary.append({
        "model": tag.replace("_", "/"),
        "avg_cer": float(df_cmp[f"cer__{tag}"].mean()),
        "avg_wer": float(df_cmp[f"wer__{tag}"].mean()),
    })
df_summary = pd.DataFrame(summary).sort_values("avg_cer")

print(df_summary)
df_cmp_view.head(10)


In [ ]:
from pathlib import Path
import glob, os
from IPython.display import Audio, display

# 파일 스캔은 1회만
audio_files_all = [Path(p) for p in glob.glob(os.path.join(DATA_AUDIO_DIR, "**", "*.flac"), recursive=True)]
wav_map = {p.name: str(p) for p in audio_files_all}


In [ ]:
from IPython.display import Audio, display

def resolve_wav_path(p):
    # p가 이미 full path이면 그대로
    if p and os.path.exists(p):
        return p
    # basename이면 map에서 찾기
    base = os.path.basename(str(p))
    return wav_map.get(base)

# 가장 성능 좋은 모델(평균 CER 최소) tag
best_tag = df_summary.iloc[0]["model"].replace("/", "_")
cer_col = f"cer__{best_tag}"

# CER 큰 순으로 5개
df_pick = df_cmp_view.sort_values(cer_col, ascending=False).head(5)

for _, r in df_pick.iterrows():
    print("=" * 90)
    print("FILE:", r["file"])
    print("PATH:", r["wav_path"])
    print("REF :", r["ref"])
    print("-" * 90)

    # 모델별 출력(현재 df_cmp_view 컬럼 순서가 이미 성능 순 정렬됨)
    for col in [c for c in df_cmp_view.columns if c.startswith("hyp__")]:
        tag = col.replace("hyp__", "")
        print(f"[{tag.replace('_','/')}] CER={r.get('cer__'+tag):.4f}  WER={r.get('wer__'+tag):.4f}")
        print(r[col])
        print()
    wav_path = resolve_wav_path(r["wav_path"])
    display(Audio(filename=wav_path))
